In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,log_loss

In [13]:
df = pd.read_csv('../0.dataset/dataset_serangan_jantung_clean.csv')
df_x = df.drop(columns='Serangan_Jantung')
df_y = df['Serangan_Jantung']

# 1.Binary Classification Support Vector Machines

In [14]:
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

In [15]:
params = {
    'kernel': ['rbf'],
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1]
}

svm = SVC()
random_search = RandomizedSearchCV(estimator=svm,param_distributions=params,n_iter=16,
                                   cv=6,scoring='accuracy',random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train)
best_model_svm = random_search.best_estimator_

sfs_svm = SequentialFeatureSelector(estimator=best_model_svm,n_features_to_select=5,direction='forward')
sfs_svm.fit(X_train,y_train)

X_train_selected = sfs_svm.transform(X_train)
X_test_selected = sfs_svm.transform(X_test)

fitur_terpilih = X_train.columns[sfs_svm.get_support()]
best_model_svm.fit(X_train_selected,y_train)

print(f'\nHyperparameter Terbaik: {random_search.best_params_}')
print(f'\nAkurasi Validasi Terbaik: {random_search.best_score_*100:.2f}%')
print(f'\nFitur Terbaik Yang Terpilih:\n{list(fitur_terpilih)}')


Hyperparameter Terbaik: {'kernel': 'rbf', 'gamma': 'scale', 'C': 1}

Akurasi Validasi Terbaik: 77.75%

Fitur Terbaik Yang Terpilih:
['Tipe_Nyeri_Dada', 'Tekanan_Darah_Rest', 'Detak_Jantung_Max', 'Angina_Olahraga', 'Oldpeak_ST']
